In [1]:
import torch.backends.cudnn as cudnn
cudnn.benchmark = True

In [2]:
import torch 
import torch.nn as nn
import torch.nn.functional as F

import torch 
import torch.nn as nn
import torch.nn.functional as F

class EQLRLinear(nn.Module):
    def __init__(self, in_features, out_features, lr_mul=1.0):
        super().__init__()
        self.lr_mul = lr_mul
        
        self.weight = nn.Parameter(torch.randn(out_features, in_features) / lr_mul)
        self.bias = nn.Parameter(torch.zeros(out_features))

        fan_in = in_features
        self.scale = ((2 / fan_in) ** 0.5) * lr_mul 
    
    def forward(self, x):
        weight = self.weight * self.scale
        bias = self.bias * self.lr_mul
        
        return F.linear(x, weight, bias)

In [3]:
import torch
import torch.nn as nn

class PixelNorm(nn.Module):
    def __init__(self, epsilon=1e-8):
        super().__init__()
        self.epsilon = epsilon

    def forward(self, x):

        norm = torch.rsqrt(torch.mean(x ** 2, dim=1, keepdim=True) + self.epsilon)
        return x * norm

class MappingNetwork(nn.Module):
    def __init__(self, z_dim=512, w_dim=512, depth=8, lr_mul=0.01):
        super().__init__()
        
        layers = [PixelNorm()]
        
        for i in range(depth):
            in_features = z_dim if i == 0 else w_dim
            
            layers.append(EQLRLinear(in_features, w_dim, lr_mul=lr_mul))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            
        self.mapping = nn.Sequential(*layers)

    def forward(self, z):
        return self.mapping(z)

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class EQLRConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0, bias=True):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_channels, in_channels, kernel_size, kernel_size))
        self.bias = nn.Parameter(torch.zeros(out_channels)) if bias else None
        self.stride = stride
        self.padding = padding

        self.fan_in = in_channels * kernel_size ** 2
        self.scale = (2 / self.fan_in) ** 0.5
        
    def forward(self, x):
        weight = self.weight * self.scale
        return F.conv2d(x, weight, self.bias, self.stride, self.padding)

In [5]:
import torch 
import torch.nn as nn
import torch.nn.functional as F
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# Assuming EQLRLinear is the class we perfected earlier
# from eq_lr import EQLRLinear 

class ModulatedConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, w_dim, demodulate=True):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.padding = kernel_size // 2
        self.demodulate = demodulate
        
        # 1. Base weights strictly N(0, 1) for Equalized LR
        self.weight = nn.Parameter(
            torch.randn(1, out_channels, in_channels, kernel_size, kernel_size)
        )
        
        # 2. Equalized Learning Rate scale factor (He Constant)
        fan_in = in_channels * kernel_size ** 2
        self.eq_lr_scale = (2 / fan_in) ** 0.5
        
        self.style_proj = EQLRLinear(w_dim, in_channels)

    def forward(self, x, w):
        batch, in_c, height, width = x.shape
        
      
        style = self.style_proj(w) + 1.0
        style = style.view(batch, 1, in_c, 1, 1)
      
        weight = self.eq_lr_scale * self.weight * style
        
        if self.demodulate:
            demod = torch.rsqrt(weight.pow(2).sum(dim=[2, 3, 4], keepdim=True) + 1e-8)
            weight = weight * demod
      
        weight = weight.view(batch * self.out_channels, in_c, self.kernel_size, self.kernel_size)
        
        x = x.view(1, batch * in_c, height, width)
        
        out = F.conv2d(x, weight, padding=self.padding, groups=batch)
        
        out = out.view(batch, self.out_channels, height, width)
        
        return out

In [6]:
class ToRGB(nn.Module):
    def __init__(self, in_channels, w_dim,img_channels=3):
        super().__init__()
        # 1x1 modulated convolution, NO demodulation
        self.conv = ModulatedConv2d(
            in_channels, 
            out_channels=img_channels,
            kernel_size=1, 
            w_dim=w_dim, 
            demodulate=False
        )
        self.bias = nn.Parameter(torch.zeros(1, 3, 1, 1))

    def forward(self, x, w):
        x = self.conv(x, w)
        x = x + self.bias
        return x

In [7]:
import torch 
import torch.nn as nn

class NoiseInjection(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.weight = nn.Parameter(torch.zeros(1, channels, 1, 1))

    def forward(self, image, noise=None):
        if noise is None:
            batch, _, height, width = image.shape
            noise = image.new_empty(batch, 1, height, width).normal_()
        return image + self.weight * noise

In [8]:
import torch 
import torch.nn as nn


class StyleBlock(nn.Module):
    def __init__(self, in_channels, out_channels,w_dim,kernel_size=3):
        super().__init__()
        self.conv =ModulatedConv2d(in_channels, out_channels, kernel_size,w_dim)
        self.noise_injection = NoiseInjection(out_channels)
        self.lrelu = nn.LeakyReLU(0.2)

        self.bias = nn.Parameter(torch.zeros(1, out_channels, 1, 1))

    def forward(self, x,w,noise=None):
        x = self.conv(x,w)
        x = self.noise_injection(x,noise)
        x = x + self.bias
        return self.lrelu(x)

In [9]:
import torch 
import torch.nn as nn

class DoubleStyleBlock(nn.Module):    
    def __init__(self, in_channels, out_channels,w_dim,kernel_size=3,depth=2):
        super().__init__()
        self.blocks = nn.ModuleList([
            StyleBlock(in_channels, in_channels, kernel_size,w_dim)
            for _ in range(depth-1)
        ])
        self.blocks.append(StyleBlock(in_channels, out_channels, kernel_size,w_dim))
    def forward(self, x,w,noise=None):
        for block in self.blocks:
            x = block(x,w,noise)
        return x

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SkipGenerator(nn.Module):
    def __init__(self,
                 z_dim,
                 w_dim,
                 mapping_network_depth,
                 mapping_network_lr_mul,
                 channels,
                 img_channels=3,
                 const_input_shape=(512,4,4)):
        super().__init__()
        self.channels = channels.copy()
        self.mapping_network = MappingNetwork(z_dim, w_dim, 
                                        mapping_network_depth, mapping_network_lr_mul)

        self.contant_input = nn.Parameter(torch.randn(1,*const_input_shape))

        self.blocks = nn.ModuleList([StyleBlock(const_input_shape[0], channels[0],w_dim)])
        self.rgb_blocks = nn.ModuleList([ToRGB(channels[0], w_dim,img_channels)])
    
        for i in range(1,len(channels)):
            self.blocks.append(DoubleStyleBlock(channels[i-1], channels[i],w_dim))
            self.rgb_blocks.append(ToRGB(channels[i], w_dim,img_channels))
        
    def forward(self, z):
        batch_size = z.shape[0]
        w = self.mapping_network(z)
        
        x = self.contant_input.repeat(batch_size, 1, 1, 1)

        x = self.blocks[0](x, w)
        rgb = self.rgb_blocks[0](x, w)

        for i in range(1,len(self.blocks)):
            x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
            x = self.blocks[i](x, w)

            rgb = F.interpolate(rgb, scale_factor=2, mode='bilinear', align_corners=False)
            rgb += self.rgb_blocks[i](x, w)
        return rgb
        
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

gen = SkipGenerator(512, 512,8, 0.01, [512,256], img_channels=3).to(device)

z= torch.randn(4,512).to(device)
gen(z).shape